In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

warnings.filterwarnings('ignore')

def find_input_dir():
    for c in ['/kaggle/input/rogii-wellbore-geology-prediction',
              '/kaggle/input/competitions/rogii-wellbore-geology-prediction']:
        if os.path.isdir(c):
            return c
    hits = glob.glob('/kaggle/input/**/sample_submission.csv', recursive=True)
    if hits:
        return os.path.dirname(hits[0])
    raise FileNotFoundError('데이터 경로를 찾을 수 없음')

INPUT_DIR = find_input_dir()
TRAIN_DIR = os.path.join(INPUT_DIR, 'train')
TEST_DIR  = os.path.join(INPUT_DIR, 'test')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'INPUT_DIR = {INPUT_DIR}')
print(f'device = {device}  ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"})')

_train_files = sorted(glob.glob(os.path.join(TRAIN_DIR, '*__horizontal_well.csv')))
TRAIN_WIDS = [os.path.basename(f).split('__')[0] for f in _train_files]
_test_files = sorted(glob.glob(os.path.join(TEST_DIR, '*__horizontal_well.csv')))
TEST_WIDS = [os.path.basename(f).split('__')[0] for f in _test_files]
print(f'Train wells: {len(TRAIN_WIDS)}')

def load_well(wid, split='train'):
    base = TRAIN_DIR if split == 'train' else TEST_DIR
    hw = pd.read_csv(os.path.join(base, f'{wid}__horizontal_well.csv'))
    tw = pd.read_csv(os.path.join(base, f'{wid}__typewell.csv'))
    return hw, tw

INPUT_DIR = /kaggle/input/competitions/rogii-wellbore-geology-prediction
device = cuda  (Tesla T4)
Train wells: 773


In [ ]:
def run_particle_filter(hw, tw, n_particles=2000, seed=42):
    """
    GR 매칭 기반 Particle Filter (baseline 파라미터).
    """
    tw_s = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    kn = hw[hw['TVT_input'].notna()]
    ev = hw[hw['TVT_input'].isna()]
    if len(ev) == 0:
        return hw['TVT_input'].values.astype(float).copy(), 0.0

    last_tvt = float(kn.iloc[-1]['TVT_input'])
    last_Z   = float(kn.iloc[-1]['Z'])
    last_MD  = float(kn.iloc[-1]['MD'])

    # GR 잡음 계산 (known 구간의 GR 잔차 std)
    tw_at_k = np.interp(kn['TVT_input'].values, tw_tvt, tw_gr)
    gs = float(np.clip(np.nanstd(kn['GR'].fillna(0).values - tw_at_k), 10., 60.))

    # 최근 30개 지점에서 진행률(rate) 계산: 깊이 변화 속도
    tail = kn.tail(30)
    dt = np.diff(tail['TVT_input'].values)
    dz = np.diff(tail['Z'].values)
    dm = np.diff(tail['MD'].values)
    m  = dm > 0
    ir = float(np.median((dt + dz)[m] / dm[m])) if m.sum() >= 3 else 0.0

    # === Motion / observation 파라미터 (baseline) ===
    MOM    = 0.998   # momentum
    VN     = 0.002   # velocity noise
    PN     = 0.005   # position noise
    RP     = 0.1     # resample 후 position perturbation
    RR     = 0.001   # resample 후 rate perturbation
    RESAMP = 0.5     # n_eff < RESAMP*N 일 때 resample

    # 가상의 드릴 N개 초기화
    N = n_particles
    rng = np.random.default_rng(seed)
    ls  = last_tvt + last_Z
    pos  = ls + 2.0 * rng.standard_normal(N)
    rate = ir + 0.01 * rng.standard_normal(N)
    w    = np.ones(N) / N

    md_v = ev['MD'].values.astype(float)
    z_v  = ev['Z'].values.astype(float)
    gr_interp = hw['GR'].interpolate(limit_direction='both').fillna(tw_gr.mean())
    gr_v = gr_interp.values.astype(float)[ev.index]

    out_vals = hw['TVT_input'].values.astype(float).copy()
    res = np.empty(len(ev))
    prev_MD = last_MD
    log_lik = 0.0

    for i in range(len(ev)):
        dm_step = max(md_v[i] - prev_MD, 1.0)
        rate = MOM * rate + VN * rng.standard_normal(N)
        pos  = pos + rate * dm_step + PN * rng.standard_normal(N)
        tvt_p = pos - z_v[i]
        tvt_p = np.clip(tvt_p, tw_tvt[0] - 100, tw_tvt[-1] + 100)
        pos   = tvt_p + z_v[i]

        # Likelihood: GR mismatch에 Gaussian
        eg = np.interp(tvt_p, tw_tvt, tw_gr)
        d  = (gr_v[i] - eg) / gs
        lk = np.maximum(np.exp(-0.5 * np.minimum(d**2, 600.)), 1e-300)

        avg_lk = float((w * lk).sum())
        log_lik += np.log(max(avg_lk, 1e-300))
        w *= lk
        ws = w.sum()
        w = w / ws if ws > 0 else np.ones(N) / N

        # Resampling
        if 1.0 / (w**2).sum() < RESAMP * N:
            cum = np.cumsum(w)
            u0  = rng.uniform(0, 1.0 / N)
            idx = np.clip(np.searchsorted(cum, u0 + np.arange(N) / N), 0, N - 1)
            pos  = pos[idx]  + RP * rng.standard_normal(N)
            rate = rate[idx] + RR * rng.standard_normal(N)
            w    = np.ones(N) / N

        res[i] = float(np.dot(w, pos - z_v[i]))
        prev_MD = md_v[i]

    out_vals[list(ev.index)] = res
    return out_vals, log_lik

def run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=16,
                        target_n_eff_ratio=0.3, return_internals=False):
    """
    여러 seed로 PF 돌리고 log-likelihood 가중평균.
    """
    preds, liks = [], []
    for s in range(n_seeds):
        p, ll = run_particle_filter(hw, tw, n_particles=n_particles, seed=s)
        preds.append(p)
        liks.append(ll)

    liks = np.array(liks)
    scale = 5.0
    weights = np.exp((liks - liks.max()) / scale)
    weights /= weights.sum()
    ensemble_pred = (weights[:, None] * np.stack(preds, 0)).sum(0)
    if return_internals:
        return ensemble_pred, liks, weights
    return ensemble_pred

In [ ]:
import pickle, time

N_WELLS       = 773                             
PF_SEEDS      = 8                                
PF_CACHE_PATH = f'/kaggle/working/pf_cache_n{N_WELLS}_s{PF_SEEDS}.pkl'

def build_pf_cache(wids):
    if os.path.exists(PF_CACHE_PATH):            
        with open(PF_CACHE_PATH, 'rb') as f:
            return pickle.load(f)
    cache, t0 = {}, time.time()
    for i, wid in enumerate(wids):
        hw, tw = load_well(wid, 'train')
        cache[wid] = run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=PF_SEEDS)
        if (i + 1) % 20 == 0:
            print(f'  PF {i+1}/{len(wids)}  ({time.time()-t0:.0f}s)')
    with open(PF_CACHE_PATH, 'wb') as f:
        pickle.dump(cache, f)
    print(f'PF 캐시 완료: {len(cache)} wells, {time.time()-t0:.0f}s')
    return cache

PF_CACHE = build_pf_cache(TRAIN_WIDS[:N_WELLS])

  PF 20/773  (69s)
  PF 40/773  (146s)
  PF 60/773  (230s)
  PF 80/773  (314s)
  PF 100/773  (397s)
  PF 120/773  (477s)
  PF 140/773  (558s)
  PF 160/773  (632s)
  PF 180/773  (712s)
  PF 200/773  (784s)
  PF 220/773  (860s)
  PF 240/773  (943s)
  PF 260/773  (1025s)
  PF 280/773  (1105s)
  PF 300/773  (1178s)
  PF 320/773  (1269s)
  PF 340/773  (1357s)
  PF 360/773  (1431s)
  PF 380/773  (1513s)
  PF 400/773  (1599s)
  PF 420/773  (1680s)
  PF 440/773  (1752s)
  PF 460/773  (1836s)
  PF 480/773  (1915s)
  PF 500/773  (1996s)
  PF 520/773  (2074s)
  PF 540/773  (2155s)
  PF 560/773  (2240s)
  PF 580/773  (2320s)
  PF 600/773  (2404s)
  PF 620/773  (2483s)
  PF 640/773  (2564s)
  PF 660/773  (2651s)
  PF 680/773  (2724s)
  PF 700/773  (2813s)
  PF 720/773  (2892s)
  PF 740/773  (2964s)
  PF 760/773  (3049s)
PF 캐시 완료: 773 wells, 3097s


In [ ]:
FEAT_COLS = ['gr', 'md_rel', 'z_rel', 'z_grad', 'tvt_in_rel', 'is_known', 'pred_pf_rel',
             'misfit_m20', 'misfit_m10', 'misfit_m5', 'misfit_0',
             'misfit_p5', 'misfit_p10', 'misfit_p20',  
             'twgr_slope']                            


def build_well_arrays(hw, tw, pred_pf):  
    df = hw.reset_index(drop=True)
    known = df['TVT_input'].notna().values
    if known.sum() == 0 or (~known).sum() == 0:
        return None

    ps   = np.where(known)[0][-1]
    md0  = float(df['MD'].iloc[ps]); z0 = float(df['Z'].iloc[ps])
    tvt0 = float(df['TVT_input'].iloc[ps])

    gr = df['GR'].interpolate(limit_direction='both').fillna(df['GR'].mean()).values
    md = df['MD'].values.astype(float); z = df['Z'].values.astype(float)

    # typewell을 TVT 순으로 정렬
    tw_s   = tw.sort_values('TVT')
    tw_tvt = tw_s['TVT'].values.astype(float)
    tw_gr  = tw_s['GR'].fillna(tw_s['GR'].mean()).values.astype(float)

    X = np.zeros((len(df), len(FEAT_COLS)), dtype=np.float32)
    X[:, 0] = gr / 100.0
    X[:, 1] = (md - md0) / 1000.0
    X[:, 2] = (z - z0) / 100.0
    X[:, 3] = np.gradient(z, md)
    tvt_in  = df['TVT_input'].fillna(tvt0).values.astype(float)
    X[:, 4] = (tvt_in - tvt0) / 100.0
    X[:, 5] = known.astype(np.float32)
    X[:, 6] = (pred_pf - tvt0) / 100.0

    # === 타입웰 윈도우 불일치 ===
    OFFS = [-20, -10, -5, 0, 5, 10, 20]
    for k, off in enumerate(OFFS):
        twgr_off  = np.interp(pred_pf + off, tw_tvt, tw_gr)  
        X[:, 7 + k] = (gr - twgr_off) / 100.0               

    # === 타입웰 GR 기울기 (오르막/내리막 방향) ===
    twgr_up = np.interp(pred_pf + 5, tw_tvt, tw_gr)
    twgr_dn = np.interp(pred_pf - 5, tw_tvt, tw_gr)
    X[:, 14] = (twgr_up - twgr_dn) / 100.0

    if 'TVT' in df.columns:
        y = ((df['TVT'].values - pred_pf) / 100.0).astype(np.float32)
    else:
        y = np.zeros(len(df), dtype=np.float32)

    return X, y, known, tvt0

In [ ]:
class WellDataset(Dataset):
    def __init__(self, wids, pf_cache):                   # ← pf_cache 인자 추가
        self.items = []
        for wid in wids:
            if wid not in pf_cache:
                continue
            hw, tw = load_well(wid, 'train')
            r = build_well_arrays(hw, tw, pf_cache[wid])       # ← PF 예측 전달
            if r is not None:
                X, y, known, tvt0 = r
                self.items.append((wid, X, y, known, tvt0))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        wid, X, y, known, tvt0 = self.items[i]
        return (torch.from_numpy(X), torch.from_numpy(y),
                torch.from_numpy(known), wid, tvt0)

In [ ]:
class TVTSeq2Seq(nn.Module):
    def __init__(self, n_feat, hidden=128, layers=2, dropout=0.2):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(n_feat, hidden, kernel_size=7, padding=3), nn.ReLU(),
            nn.Conv1d(hidden, hidden, kernel_size=7, padding=3), nn.ReLU(),
        )
        self.lstm = nn.LSTM(hidden, hidden, num_layers=layers, batch_first=True,
                            bidirectional=True, dropout=dropout)
        self.head = nn.Linear(hidden * 2, 1)

    def forward(self, x):                       # x: (B, L, F)
        h = self.cnn(x.transpose(1, 2)).transpose(1, 2)   # (B, F, L)
        h, _ = self.lstm(h)                     # (B, L, hidden*2)
        return self.head(h).squeeze(-1)

In [ ]:
from torch.nn.utils.rnn import pad_sequence

def collate(batch):
    # batch = [(X, y, known, wid, tvt0), ...]
    Xs, ys, knowns, wids, tvt0s = zip(*batch)
    lengths = [x.shape[0] for x in Xs]  

    Xp = pad_sequence(Xs,     batch_first=True)             # (B, Lmax, F)
    yp = pad_sequence(ys,     batch_first=True)             # (B, Lmax)
    kp = pad_sequence(knowns, batch_first=True)             # (B, Lmax)

    # valid: 진짜 데이터=True, 패딩=False
    Lmax  = Xp.shape[1]
    valid = torch.zeros((len(batch), Lmax), dtype=torch.bool)
    for i, L in enumerate(lengths):
        valid[i, :L] = True

    return Xp, yp, kp, valid, list(wids), torch.tensor(tvt0s, dtype=torch.float32)

In [ ]:
N_WELLS  = 773   
LAM      = 0.1      
MAX_EPOCH = 100
PATIENCE  = 15

# 우물 단위로 train/val 분리
rng = np.random.default_rng(42)
wids = list(TRAIN_WIDS[:N_WELLS])
rng.shuffle(wids)
n_val = int(len(wids) * 0.2)
val_wids, tr_wids = wids[:n_val], wids[n_val:]

tr_dl = DataLoader(WellDataset(tr_wids, PF_CACHE), batch_size=8, shuffle=True,  collate_fn=collate)
va_dl = DataLoader(WellDataset(val_wids, PF_CACHE), batch_size=8, shuffle=False, collate_fn=collate)

model = TVTSeq2Seq(n_feat=len(FEAT_COLS),
                  hidden=256, layers=3, dropout=0.3).to(device)
opt   = torch.optim.Adam(model.parameters(), lr=1e-3)
sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=4, factor=0.5)

def compute_loss(pred, y, known, valid):
    hidden = valid & (~known)              
    mse = ((pred - y)[hidden] ** 2).mean()
    d = pred[:, 1:] - pred[:, :-1]
    vv = valid[:, 1:] & valid[:, :-1]
    smooth = (d[vv] ** 2).mean()
    return mse + LAM * smooth

best, patience = 1e9, 0
for epoch in range(MAX_EPOCH):
    model.train()
    for Xp, yp, kp, valid, _, _ in tr_dl:
        Xp, yp, kp, valid = Xp.to(device), yp.to(device), kp.to(device), valid.to(device)
        pred = model(Xp)
        loss = compute_loss(pred, yp, kp, valid)
        opt.zero_grad(); loss.backward(); opt.step()

    # validation: hidden TVT를 ft로 복원해 RMSE
    model.eval(); se, cnt = 0.0, 0
    with torch.no_grad():
        for Xp, yp, kp, valid, _, _ in va_dl:
            Xp, yp, kp, valid = Xp.to(device), yp.to(device), kp.to(device), valid.to(device)
            pred = model(Xp)
            hidden = valid & (~kp)
            err = (pred - yp)[hidden] * 100.0
            se  += (err ** 2).sum().item(); cnt += hidden.sum().item()
    val_rmse = (se / cnt) ** 0.5
    sched.step(val_rmse)
    print(f'epoch {epoch:>2}  val_RMSE = {val_rmse:.3f} ft')

    if val_rmse < best:
        best, patience = val_rmse, 0
        torch.save(model.state_dict(), '/kaggle/working/seq2seq_best.pt')
    else:
        patience += 1
        if patience >= PATIENCE:
            print('early stopping'); break

print(f'\nbest val RMSE = {best:.3f} ft   (baseline PF ≈ 7~11 ft 와 비교)')

In [ ]:
# val 우물에서 PF baseline RMSE
pf_se, pf_cnt = 0.0, 0
for wid in val_wids:
    hw, _  = load_well(wid, 'train')
    mask   = hw['TVT_input'].isna().values
    y_true = hw['TVT'].values[mask]
    pred_pf = PF_CACHE[wid][mask]
    pf_se  += ((y_true - pred_pf) ** 2).sum()
    pf_cnt += mask.sum()
print(f'val 우물 PF baseline RMSE = {(pf_se / pf_cnt) ** 0.5:.3f} ft')

val 우물 PF baseline RMSE = 10.419 ft


In [ ]:
import torch, glob

# 1) 학습한 best 모델 로드
ckpt = '/kaggle/input/datasets/yejinkyo/seq2seq/seq2seq_best.pt'
model = TVTSeq2Seq(n_feat=len(FEAT_COLS), hidden=256, layers=3).to(device)
model.load_state_dict(torch.load(ckpt, map_location=device))
model.eval()

print(f'로드: {ckpt}')

# 2) sample_submission 파싱
sample = pd.read_csv(os.path.join(INPUT_DIR, 'sample_submission.csv'))
sample['well']    = sample['id'].str[:8]
sample['row_idx'] = sample['id'].str[9:].astype(int)

rows = []
for wid in TEST_WIDS:
    hw, tw = load_well(wid, 'test')

    # 3) seq2seq 입력용 PF 예측 (학습 때와 같은 seed 수로 일관성 유지)
    pred_pf = run_pf_lik_ensemble(hw, tw, n_particles=500, n_seeds=16)

    # 4) feature 만들고 모델 추론 (배치 1개)
    X, _, known, tvt0 = build_well_arrays(hw, tw, pred_pf)
    Xt = torch.from_numpy(X).unsqueeze(0).to(device)        # (1, L, F)
    with torch.no_grad():
        out = model(Xt).squeeze(0).cpu().numpy()            # (L,)

    # 5) 잔차 복원: 최종 TVT = PF예측 + 모델보정×100
    pred_tvt = pred_pf + out * 100.0

    # 6) sample의 해당 행에 채우기
    ws = sample[sample['well'] == wid]
    for _, r in ws.iterrows():
        ridx = int(r['row_idx'])
        rows.append({'id': r['id'], 'tvt': float(pred_tvt[ridx])})
    print(f'{wid}: {len(ws)} rows')

submission = pd.DataFrame(rows)
submission = sample[['id']].merge(submission, on='id', how='left')

submission.to_csv('submission.csv', index=False)
print(f'\n저장 완료: {len(submission)} rows, 결측 {submission["tvt"].isna().sum()}')
print(submission.head())

로드: /kaggle/working/seq2seq_best.pt
000d7d20: 3836 rows
00bbac68: 6014 rows
00e12e8b: 4301 rows

저장 완료: 14151 rows, 결측 0
              id           tvt
0  000d7d20_1442  11747.905614
1  000d7d20_1443  11747.967693
2  000d7d20_1444  11747.985643
3  000d7d20_1445  11747.998499
4  000d7d20_1446  11748.026471
